In [ ]:
%env AG_DATA=/content/ag_data
!mkdir -p $AG_DATA/ref $AG_DATA/weights
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# torch runtime (the port); no jax here. clone once, pull after every local push
!git -C /content/ag pull -q 2>/dev/null || git clone -q -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag
!pip install -q -e /content/ag pyfaidx
%cd /content/ag/experiments
!git log -1 --oneline

In [ ]:
# same reference genome + index as the golden run
!test -f $AG_DATA/ref/GRCh38.p13.genome.fa || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa
!test -s $AG_DATA/ref/GRCh38.p13.genome.fa.fai || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa.fai https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa.fai
!ls -lh $AG_DATA/ref

In [ ]:
# step 1: golden input rebuilt with the port, must match both hashes
!python scripts/port_input.py -v

In [ ]:
# step 2: pass 1, module tree with shapes -> results/modules_131k.txt
!python scripts/trace_modules.py --check -v

In [ ]:
# step 3: pass 2, perfetto trace -> results/trace_131k.json
!python scripts/trace_kernels.py -v

In [ ]:
!ls -lh results
!head -30 results/modules_131k.txt